# In a nutshell

Experiment with prompts and different models on a subset of the annotated dataset.

# Setup

In [1]:
import json
import os
import sys
from pathlib import Path

# Adjust working dir/Python path
# exactly once per session
# using IPython user namespace for run-once behaviour.
ipy = get_ipython()
if not ipy.user_ns.get("syspath_set", False):
    os.chdir("..")
    sys.path.append(".")
    ipy.user_ns["syspath_set"] = True

In [ ]:
Path.cwd()

In [3]:
from dotenv import load_dotenv

# Load API key from .env file if present.
load_dotenv()

# Not recommended: set API key manually
# %env VAR_NAME=value

True

# Load the data

In [4]:
dataset_path = "data/eng_Latn_sample_for_llm_eval.parquet"

In [5]:
from datasets import Dataset

ds = Dataset.from_parquet(dataset_path)
print(f"Dataset size: {len(ds)} docs")
print(f"Columns:{ds.column_names}")
print(ds[:1])

Dataset size: 25 docs
Columns:['task_id', 'file_name', 'html', 'language', 'annotations', 'annotation_count', 'filename_warc', 'url', 'timestamp', 'collection']
{'task_id': [181409468], 'file_name': ['0ac1c6fa-57_ia_o_nz.html'], 'html': ['<!DOCTYPE html PUBLIC "-//W3C//DTD XHTML 1.0 Strict//EN" "http://www.w3.org/TR/xhtml1/DTD/xhtml1-strict.dtd">\n<html xmlns="http://www.w3.org/1999/xhtml"\nxml:lang="en" xmlns:fb="http://www.facebook.com/2008/fbml"\nxmlns:og="http://ogp.me/ns#" lang="en">\n<head>\n<meta http-equiv="Content-Type" content="text/html; charset=utf-8" />\n<title>ApplePlus iVehicle Accessories</title>\n<meta name="description" content="Get your car to be friend with your i-Devices. We\'ll build the bridge for you!" />\n<meta name="keywords" content="iPhone,iPad,iPod,accessories,iPad cases" />\n<meta name="robots" content="INDEX,FOLLOW" />\n<link rel="icon" href="http://www.appleplus.co.nz/media/favicon/default/favicon.ico" type="image/x-icon" />\n<link rel="shortcut icon" hr

# Experiment settings

In [6]:
config = {
    "name": "deepseek-r1-distill-llama-70b-dev",
    "model": "deepseek/deepseek-r1-distill-llama-70b",
    "connector": "openrouter",
    "concurrent_requests": 4,
    "prompt": {
        "system": "Act as a veteran dataset annotator working for OpenAI. Your task is to annotate all main content text in given html. Output the identified main content text as JSONL object with a single key 'annotations'. The value of 'annotations' is a list in which each object is a single main content text span with a single field 'text'. The value of 'text' is the literal value of the identified main content text span.\n Hints:\n        - Main content text must be distinct from repetitive boilerplate and provide unique or meaningful information relevant to the page.\n        - Main content text includes headings, dates, locations, tables, comments, lists of items, item properties and specifications, item prices, item reviews, image captions, forum threads.\n        - Main content text generally excludes common boilerplate elements that appear on every page of a website such as navigation menus, sorting dropdowns, headers, and footers.\n        - Always try to annotate the longest continuous span.\n        - Annotate only the spans that you are sure about. If you are not sure about an annotation, skip it.\n        - If there is nothing to annotate in the html, 'annotations' will be an empty list.\n        - Respond with a plain Python-compatible string instantly pluggable into json.loads. Do not format or pretty-print the output. \n        - All fields are compulsory. Output nothing but the JSONL object.",
        "user": "Given the html: \n ```{html}```\n, annotate the main content text.",
    },
}

# Call LLM annotator


In [17]:
import subprocess

args = [
    "--input",
    dataset_path,
    "--max_docs",
    "5",
    "--config",
    json.dumps(config),
]

process = subprocess.Popen(
    ["python", "-u", "-m", "llm.llm_annotator", *args],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
)

for line in process.stdout:
    print(line, end="")

process.wait()

15:59:15 [INFO] Loaded config {'name': 'deepseek-r1-distill-llama-70b-dev', 'model': 'deepseek/deepseek-r1-distill-llama-70b', 'connector': 'openrouter', 'concurrent_requests': 4, 'prompt': {'system': "Act as a veteran dataset annotator working for OpenAI. Your task is to annotate all main content text in given html. Output the identified main content text as JSONL object with a single key 'annotations'. The value of 'annotations' is a list in which each object is a single main content text span with a single field 'text'. The value of 'text' is the literal value of the identified main content text span.\n Hints:\n        - Main content text must be distinct from repetitive boilerplate and provide unique or meaningful information relevant to the page.\n        - Main content text includes headings, dates, locations, tables, comments, lists of items, item properties and specifications, item prices, item reviews, image captions, forum threads.\n        - Main content text generally exclu

0

# Load/merge anotations

In [8]:
result_path = Path(f"llm/.annotations/{config['name']}/results.jsonl")

with result_path.open(encoding="utf-8") as f:
    llm_annotations = [json.loads(line) for line in f]

print(f"Total docs annotated: {len(llm_annotations)}")

Total docs annotated: 23


In [9]:
import polars as pl

llm_annotations = pl.DataFrame(llm_annotations)
llm_annotations.head(1)

timestamp,config,model,task_id,duration_ms,annotations,total_tokens
str,str,str,i64,i64,list[struct[1]],i64
"""17/11/2025 19:03:54""","""deepseek-r1-distill-llama-70b""","""deepseek/deepseek-r1-distill-l…",181409468,203163,"[{""Get your car to be friend with your i-Devices. We'll build the bridge for you!""}, {""SGP Mobile Stand for Smart Phones""}, … {""$45.00""}]",18241


In [ ]:
from bs4 import BeautifulSoup


def strip_html(text: str) -> str:
    soup = BeautifulSoup(text, "html.parser")

    return soup.get_text(separator=" ", strip=True)


def merge_annotations(x: list[dict[str, str]]) -> str:
    return " ".join([strip_html(k["text"]) for k in x])


llm_annotations = llm_annotations.with_columns(
    pl.col("annotations")
    .map_elements(lambda x: merge_annotations(x), return_dtype=pl.Utf8)
    .alias("annotations_as_string")
)

with pl.Config(fmt_str_lengths=10**5, fmt_table_cell_list_len=10**5, tbl_cols=-1, tbl_rows=-1):
    print(llm_annotations.head(1))

shape: (1, 8)
┌────────────┬────────────┬────────────┬───────────┬───────────┬───────────┬───────────┬───────────┐
│ timestamp  ┆ config     ┆ model      ┆ task_id   ┆ duration_ ┆ annotatio ┆ total_tok ┆ annotatio │
│ ---        ┆ ---        ┆ ---        ┆ ---       ┆ ms        ┆ ns        ┆ ens       ┆ ns_as_str │
│ str        ┆ str        ┆ str        ┆ i64       ┆ ---       ┆ ---       ┆ ---       ┆ ing       │
│            ┆            ┆            ┆           ┆ i64       ┆ list[stru ┆ i64       ┆ ---       │
│            ┆            ┆            ┆           ┆           ┆ ct[1]]    ┆           ┆ str       │
╞════════════╪════════════╪════════════╪═══════════╪═══════════╪═══════════╪═══════════╪═══════════╡
│ 17/11/2025 ┆ deepseek-r ┆ deepseek/d ┆ 181409468 ┆ 203163    ┆ [{"Get    ┆ 18241     ┆ Get your  │
│ 19:03:54   ┆ 1-distill- ┆ eepseek-r1 ┆           ┆           ┆ your car  ┆           ┆ car to be │
│            ┆ llama-70b  ┆ -distill-l ┆           ┆           ┆ to be     ┆ 

# Compute metrics

In [11]:
from d2g_evaluation.evaluation.evaluate_human_vs_tool import HumanVsToolEvaluation

annotated_ids = set(llm_annotations["task_id"])
# sort both datasets by id, so that new llm annotation column is added correctly
ds_with_llm_annotations = ds.filter(lambda x: x["task_id"] in annotated_ids).sort(column_names=["task_id"])
llm_annotations = llm_annotations.sort(["task_id"])
llm_annotations_as_list = llm_annotations["annotations"].list.eval(pl.element().struct.field("text")).to_list()
ds_with_llm_annotations = ds_with_llm_annotations.add_column(
    "llm_annotations_as_string",
    llm_annotations["annotations_as_string"],
).add_column("llm_annotations", llm_annotations_as_list)


metrics = {
    "metric_name": "lcs_token_matching",
    "string_preprocessing_method": "normalize_string",
    "tokenization_method": "char_ngrams",
    "n": 3,
    "is_symmetric_forced": False,
}

eval_tool = HumanVsToolEvaluation()
eval_tool.INSUFFICIENT_ANNOTATIONS = 1

eval_result = eval_tool.evaluate(
    dataset=ds_with_llm_annotations,
    tool_column="llm_annotations_as_string",
    tool_name=llm_annotations[0]["model"],
    **metrics,
)

Evaluating with metric 'lcs_token_matching': 100%|##########| 23/23 [00:00<?, ? examples/s]

In [12]:
eval_result.column_names

['task_id',
 'file_name',
 'html',
 'language',
 'annotations',
 'annotation_count',
 'filename_warc',
 'url',
 'timestamp',
 'collection',
 'llm_annotations_as_string',
 'llm_annotations',
 'sample_result_human_vs_tool',
 'pairwise_results_human_vs_tool']

In [13]:
eval_result_df = eval_result.to_polars()
eval_stats = eval_result_df.select(
    [
        pl.col("task_id"),
        pl.col("sample_result_human_vs_tool")
        .struct.field("precision")
        .struct.field("mean")
        .round(3)
        .alias("precision"),
        pl.col("sample_result_human_vs_tool").struct.field("recall").struct.field("mean").round(3).alias("recall"),
        pl.col("sample_result_human_vs_tool").struct.field("f1").struct.field("mean").round(3).alias("f1"),
        pl.col("llm_annotations"),
        pl.col("llm_annotations_as_string"),
        pl.col("annotations").list.eval(pl.element().struct[8].list[0].struct["text"]),
    ],
)

mean_p = round(eval_stats["precision"].mean(), 3)
mean_r = round(eval_stats["recall"].mean(), 3)
mean_f1 = round(eval_stats["f1"].mean(), 3)


# show all rows
pl.Config.set_tbl_rows(-1)
print(eval_stats[["task_id", "precision", "recall", "f1"]])
print(f"AVG:         {mean_p: < 10} |{mean_r: < 7} | {mean_f1: < 10}")

shape: (23, 4)
┌───────────┬───────────┬────────┬───────┐
│ task_id   ┆ precision ┆ recall ┆ f1    │
│ ---       ┆ ---       ┆ ---    ┆ ---   │
│ i64       ┆ f64       ┆ f64    ┆ f64   │
╞═══════════╪═══════════╪════════╪═══════╡
│ 181402115 ┆ 0.994     ┆ 0.162  ┆ 0.278 │
│ 181402673 ┆ 0.996     ┆ 0.822  ┆ 0.9   │
│ 181402851 ┆ 1.0       ┆ 0.945  ┆ 0.971 │
│ 181405856 ┆ 0.406     ┆ 0.959  ┆ 0.568 │
│ 181405931 ┆ 1.0       ┆ 0.847  ┆ 0.917 │
│ 181406292 ┆ 0.861     ┆ 1.0    ┆ 0.924 │
│ 181409318 ┆ 0.238     ┆ 0.949  ┆ 0.381 │
│ 181409340 ┆ 0.818     ┆ 0.846  ┆ 0.831 │
│ 181409392 ┆ 1.0       ┆ 1.0    ┆ 1.0   │
│ 181409394 ┆ 0.923     ┆ 0.813  ┆ 0.863 │
│ 181409424 ┆ 0.724     ┆ 0.542  ┆ 0.62  │
│ 181409430 ┆ 0.989     ┆ 0.744  ┆ 0.849 │
│ 181409468 ┆ 0.825     ┆ 0.559  ┆ 0.667 │
│ 181409541 ┆ 0.664     ┆ 0.976  ┆ 0.79  │
│ 181409543 ┆ 1.0       ┆ 1.0    ┆ 1.0   │
│ 181410937 ┆ 0.983     ┆ 0.947  ┆ 0.965 │
│ 181410959 ┆ 0.709     ┆ 0.753  ┆ 0.729 │
│ 181414042 ┆ 0.78      ┆ 0.937  ┆ 0.85

In [14]:
# Document length vs token cost
mean_tokens_consumed = round(llm_annotations["total_tokens"].mean())
mean_doclength_chars = round(ds_with_llm_annotations.to_polars()["html"].map_elements(lambda x: len(x)).mean())
compression_ratio = round(mean_doclength_chars / mean_tokens_consumed, 2)

print(
    f"Avg doc length: {mean_doclength_chars:,} chars | \
Avg tokens consumed: {mean_tokens_consumed:,} | \
Compression ratio: {compression_ratio}"
)

Avg doc length: 70,684 chars | Avg tokens consumed: 21,890 | Compression ratio: 3.23


# Explore low-performers

In [19]:
low_performers = eval_stats.filter(pl.col("f1") < 0.3)  # noqa

with pl.Config(fmt_str_lengths=10**5, fmt_table_cell_list_len=10**5, tbl_cols=-1, tbl_rows=-1):
    print(low_performers["task_id", "precision", "recall", "f1", "llm_annotations", "annotations"])

shape: (1, 6)
┌───────────┬───────────┬────────┬───────┬────────────────────────────┬────────────────────────────┐
│ task_id   ┆ precision ┆ recall ┆ f1    ┆ llm_annotations            ┆ annotations                │
│ ---       ┆ ---       ┆ ---    ┆ ---   ┆ ---                        ┆ ---                        │
│ i64       ┆ f64       ┆ f64    ┆ f64   ┆ list[str]                  ┆ list[str]                  │
╞═══════════╪═══════════╪════════╪═══════╪════════════════════════════╪════════════════════════════╡
│ 181402115 ┆ 0.994     ┆ 0.162  ┆ 0.278 ┆ ["Chrome Stand", "This     ┆ ["Chrome Stand\nKent       │
│           ┆           ┆        ┆       ┆ chrome shaving brush stand ┆ Brushes\n£6\nThis chrome   │
│           ┆           ┆        ┆       ┆ is the essential tool to   ┆ shaving brush stand is the │
│           ┆           ┆        ┆       ┆ allow your shaving brush   ┆ essential tool to allow    │
│           ┆           ┆        ┆       ┆ to dry naturally.",        ┆ your 